# State Bench Long-Context: Transformer vs Hybrid vs GDN

This notebook compares matched Transformer, Hybrid, and GDN checkpoints at multiple parameter scales on the state-bench long-context tasks (results group `jacksonp-state-bench-pt`, scored against the floor-8 dataset generation). It reads per-example prediction artifacts, derives exact prompt-token length as `num_tokens_all - num_tokens`, and supports task-level and accuracy-versus-prompt-length analysis.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path

from IPython.display import display
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
FIGURES_DIR = PROJECT_ROOT / "figures"
import aesthetics as aes  # noqa: E402
import results_io as rio  # noqa: E402

# jacksonp-ai2/state-bench-lc is gated; HF_TOKEN must be set for the
# load_dataset() metadata joins below to authenticate.
load_dotenv(PROJECT_ROOT / ".env")

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

## Configuration

In [ ]:
RESULTS_ROOT = PROJECT_ROOT / "results"
# Jobs are selected rather than grouped: every matching group directory under
# results/ is scanned and the newest job wins per model/task, so a new sweep is
# picked up without naming its group here. These three knobs narrow the scan
# (all optional, and applied before any prediction file is opened):
#   RESULT_GROUPS         -- group directories or globs, None for all
#   RESULTS_SINCE         -- drop jobs older than this timestamp
#   RESULT_EXPERIMENT_IDS -- pin the analysis to specific jobs
# The glob keeps the directly-trained groups out, and the date cutoff keeps out
# the old *-lc* groups, which were scored against the pre-floor-8 dataset
# generation (and, earlier still, the coarse tokens_0_100k strata).
RESULT_GROUPS = ["*-pt*"]
RESULTS_SINCE = "2026-08-27"
RESULT_EXPERIMENT_IDS = None
TASK_RE = re.compile(
    r"state_bench_(?:lc_)?(.+?)_(aperiodic|periodic|r_trivial)(?:_tokens_.+)?$"
)
COMPLEXITY_ORDER = ["aperiodic", "periodic", "r_trivial"]
ARCHITECTURE_ORDER = ["Transformer", "Qwen", "Hybrid", "GDN"]
REFERENCE_ARCHITECTURE = "Transformer"
COMPARISON_ARCHITECTURE_ORDER = ["Hybrid", "GDN"]
MODEL_SIZE_ORDER = [275, 800, 1_400, 7_000]
MODEL_SIZE_LABELS = {275: "275M", 800: "800M", 1_400: "1.4B", 7_000: "7B"}
# Architecture determines hue; parameter count determines light-to-dark shade
# via aes.model_size_color_millions, shared with dt_results (raised floor:
# 275M is the lightest shade, larger sizes interpolate darker).
MODEL_SIZE_RE = re.compile(
    r"(?:^|[-_/])(?P<value>\d+(?:\.\d+)?)(?P<unit>[mb])(?=$|[-_/\s])",
    flags=re.IGNORECASE,
)
# Fallback only. The plots derive chance from each subset's candidate
# counts, which are two for two-variable samples and five otherwise.
CHANCE_ACCURACY = 0.20
CHANCE_LINE_COLOR = aes.CHANCE_LINE_COLOR
CONTEXT_REFERENCE_LINES = {8_000: "#2CA02C", 64_000: "#17BECF"}
MIN_BIN_EXAMPLES = 3
DEPENDENCY_LENGTH_FIELD = "max_dependency_length"
# The source HF dataset also carries per-example sensitivity metadata
# (computed over the underlying finite-state monoid), joined the same way
# as dependency length below.
SENSITIVITY_FIELDS = [
    "sensitivity_state_count",
    "sensitivity_output_count",
    "sensitivity_density",
    "sensitivity_coverage",
    "mean_live_fanout",
]
JOINED_METADATA_FIELDS = [DEPENDENCY_LENGTH_FIELD, *SENSITIVITY_FIELDS]
JOINED_METADATA_COLUMNS = ["dependency_length", *SENSITIVITY_FIELDS]
DEPENDENCY_EDGES = np.r_[-0.5, 0.5, 1.5, 2.5, 2 ** np.arange(2, 16) + 0.5]
SENSITIVITY_DENSITY_EDGES = np.linspace(0, 1, 11)


def model_architecture(result: dict) -> str:
    text = f"{result.get('experiment_name', '')} {result.get('model', '')}".lower()
    if "qwen" in text:
        return "Qwen"
    if "gdn" in text:
        return "GDN"
    if "transformer" in text:
        return "Transformer"
    if "hybrid" in text:
        return "Hybrid"
    if "olmo" in text:
        return "Transformer"
    raise ValueError(f"Unrecognized architecture: {text}")


def model_size_millions(result: dict) -> int:
    text = f"{result.get('experiment_name', '')} {result.get('model', '')}"
    match = MODEL_SIZE_RE.search(text)
    if match is None:
        raise ValueError(f"Unrecognized model size: {text}")
    value = float(match.group("value"))
    millions = value * (1_000 if match.group("unit").lower() == "b" else 1)
    if not millions.is_integer():
        raise ValueError(f"Model size is not an integer number of millions: {text}")
    return int(millions)


def model_size_label(size_millions: int) -> str:
    if size_millions in MODEL_SIZE_LABELS:
        return MODEL_SIZE_LABELS[size_millions]
    if size_millions >= 1_000:
        return f"{size_millions / 1_000:g}B"
    return f"{size_millions:g}M"


def model_variant(result: dict) -> str | None:
    """Within-family variant for the suite's non-hybrid checkpoints.

    The suite's pure transformer keeps the backbone's NoPE attention, and its
    pure GDN keeps the default allow_neg_eigval=True ("full") delta rule;
    external baselines carry no variant tag.
    """
    text = f"{result.get('experiment_name', '')} {result.get('model', '')}".lower()
    if "hybrid-small-transformer" in text:
        return "NoPE"
    if "hybrid-small-gdn" in text:
        return "full"
    return None


def model_label(architecture: str, size_millions: int, variant: str | None = None) -> str:
    size = model_size_label(size_millions)
    if variant:
        return f"{architecture} ({variant}, {size})"
    return f"{architecture} ({size})"


def nested_metric(metrics: dict, metric: str) -> float:
    value = metrics
    for key in metric.split(":"):
        value = value[key]
    return float(value)


def weighted_chance(data: pd.DataFrame | None = None) -> float:
    """Example-weighted chance accuracy over a plotted subset.

    Accepts either per-example rows (a `chance` column) or an aggregated frame
    carrying `chance_total`/`chance_examples`, and falls back to the nominal
    rate when a panel has no data.
    """
    if data is None or len(data) == 0:
        return CHANCE_ACCURACY
    if "chance_total" in data and "chance_examples" in data:
        examples = data["chance_examples"].sum()
        return data["chance_total"].sum() / examples if examples else CHANCE_ACCURACY
    if "chance" in data:
        return float(data["chance"].mean())
    return CHANCE_ACCURACY


def add_chance_line(ax: plt.Axes, data: pd.DataFrame | None = None) -> None:
    """Mark chance for the samples in `data`, falling back to the nominal rate."""
    ax.axhline(
        weighted_chance(data), color=CHANCE_LINE_COLOR, linestyle="--", linewidth=1.25
    )


def artifact_path(result: dict, task: dict, kind: str) -> Path | None:
    """Locate a task artifact inside the results group its job came from."""
    return rio.artifact_file(result, task, kind)

## Load per-example results

Runs may cover different token ranges. Prompt tokens come directly from each model output; all answer choices for an example share the same derived prompt length.

In [ ]:
# Selection runs on experiment metadata alone, so superseded predictions are
# never read off disk: load_experiments applies the group/date/id filters and
# select_latest keeps the newest job per model/task, both before the artifact
# scan and the dataset joins below.
loaded_results = rio.load_experiments(
    RESULTS_ROOT,
    groups=RESULT_GROUPS,
    since=RESULTS_SINCE,
    experiment_ids=RESULT_EXPERIMENT_IDS,
)
raw_results = rio.select_latest(loaded_results)
superseded_results = rio.superseded_table(loaded_results, raw_results)
print(
    f"Selected {len(raw_results)} of {len(loaded_results)} job(s), "
    f"{sum(len(result['tasks']) for result in raw_results):,} task results"
)
display(rio.selection_table(raw_results))
if not superseded_results.empty:
    print("Superseded by newer jobs:")
    display(superseded_results)

valid_tasks = []
skipped_tasks = []
for result in raw_results:
    for task in result.get("tasks", []):
        task_name = task.get("task", "<unnamed>")
        match = TASK_RE.fullmatch(task_name)
        if match is None:
            continue
        missing_fields = [
            field for field in ("primary_metric", "task_hash") if not task.get(field)
        ]
        request_path = artifact_path(result, task, "requests") if not missing_fields else None
        prediction_path = artifact_path(result, task, "predictions") if not missing_fields else None
        if missing_fields or request_path is None or prediction_path is None:
            reason = (
                f"missing {', '.join(missing_fields)}"
                if missing_fields
                else "missing or ambiguous request/prediction artifacts"
            )
            skipped_tasks.append((result.get("experiment_name", "<unnamed>"), task_name, reason))
            continue
        if not task.get("config", {}).get("data_source"):
            skipped_tasks.append((result.get("experiment_name", "<unnamed>"), task_name, "missing data source"))
            continue
        valid_tasks.append((result, task, match, request_path, prediction_path))

if skipped_tasks:
    skipped_by_reason = pd.Series([reason for _, _, reason in skipped_tasks]).value_counts()
    print(f"Skipped {len(skipped_tasks):,} incomplete task result(s):")
    print(skipped_by_reason.to_string())

# valid_tasks = [
#     (result, task, match, request_path, prediction_path)
#     for result, task, match, request_path, prediction_path in valid_tasks
#     if result.get("experiment_id") in valid_task_exp_ids
# ]

# The saved evaluation request documents omit dependency-length and
# sensitivity metadata, so join them from the source HF dataset by example
# ID. This applies to every complexity, not just periodic: aperiodic and
# r-trivial task configs carry the same `config.data_source` pointer.
# rio.cached_doc_metadata streams only the needed columns and caches the rows
# under results/doc_metadata keyed by the dataset's commit SHA (shared with
# dt_results), so reruns never re-resolve or re-download shards.
DOC_METADATA_CACHE_DIR = RESULTS_ROOT / "doc_metadata"
example_metadata_by_task = {}
for _, task, match, _, _ in valid_tasks:
    if task["task"] in example_metadata_by_task:
        continue
    source = task["config"]["data_source"]
    example_metadata_by_task[task["task"]] = rio.cached_doc_metadata(
        source["path"],
        source["subset"],
        source["split"],
        JOINED_METADATA_FIELDS,
        DOC_METADATA_CACHE_DIR,
        id_column="example_id",
        revision=source.get("revision"),
    )
rows = []
for result, task, match, request_path, prediction_path in valid_tasks:
    architecture = model_architecture(result)
    size_millions = model_size_millions(result)
    variant = model_variant(result)
    model = model_label(architecture, size_millions, variant)
    domain, complexity = match.groups()
    metric = task["primary_metric"]
    with request_path.open() as request_stream, prediction_path.open() as stream:
        if stream:
            for line in stream:
                prediction = json.loads(line)
                request = (
                    json.loads(next(request_stream))
                    if request_stream is not None
                    else {}
                )
                doc = request.get("doc", {})
                if request and prediction["doc_id"] != request["doc_id"]:
                    raise ValueError(
                        f"Prediction/request mismatch: {task['task']} doc {prediction['doc_id']}"
                    )
                prompt_lengths = [
                    output["num_tokens_all"] - output["num_tokens"]
                    for output in prediction["model_output"]
                ]
                if len(set(prompt_lengths)) != 1:
                    raise ValueError(
                        f"Choice-dependent prompt length: {task['task']} doc {prediction['doc_id']}"
                    )
                example_metadata = example_metadata_by_task.get(task["task"], {}).get(
                    prediction["native_id"], {}
                )
                rows.append(
                    {
                        "model": model,
                        "experiment_name": result["experiment_name"],
                        "experiment_id": result["experiment_id"],
                        "architecture": architecture,
                        "size_millions": size_millions,
                        "variant": variant,
                        "task": task["task"],
                        "domain": domain.replace("_", " ").title(),
                        "complexity": complexity.replace("_", "-"),
                        "doc_id": prediction["doc_id"],
                        "native_id": prediction["native_id"],
                        "prompt_tokens": prompt_lengths[0],
                        "chance": 1.0 / len(prediction["model_output"]),
                        "score": nested_metric(prediction["instance_metrics"], metric),
                        "dependency_length": example_metadata.get(DEPENDENCY_LENGTH_FIELD),
                        **{
                            field: example_metadata.get(field)
                            for field in SENSITIVITY_FIELDS
                        },
                        "padding_lines": doc.get("num_filler_lines"),
                        "num_variables": doc.get("num_variables"),
                        "num_assignments": doc.get("num_assignments"),
                        "padding_complexity": doc.get("filler_distraction")
                        or ("none" if doc else None),
                    }
                )
item_df = pd.DataFrame(rows)
present_sizes = sorted(set(item_df["size_millions"]))
ordered_sizes = [size for size in MODEL_SIZE_ORDER if size in present_sizes]
ordered_sizes.extend(size for size in present_sizes if size not in ordered_sizes)
model_info = (
    item_df[["model", "architecture", "size_millions", "variant"]]
    .drop_duplicates("model")
    .set_index("model")
)
architecture_rank = {arch: rank for rank, arch in enumerate(ARCHITECTURE_ORDER)}
MODEL_ORDER = sorted(
    model_info.index,
    key=lambda m: (
        architecture_rank.get(model_info.at[m, "architecture"], len(architecture_rank)),
        ordered_sizes.index(model_info.at[m, "size_millions"]),
        str(model_info.at[m, "variant"]),
    ),
)
item_df["model"] = pd.Categorical(item_df["model"], MODEL_ORDER, ordered=True)
MODEL_ARCHITECTURES = model_info["architecture"].to_dict()
MODEL_SIZES = model_info["size_millions"].to_dict()
MODEL_VARIANTS = model_info["variant"].to_dict()
# Marker encodes the within-family variant, matching dt_results: circles for
# canonical variants (RoPE positions / full delta rule), squares for the
# restricted ones (NoPE / SDP).
MODEL_MARKERS = {
    model: ("s" if MODEL_VARIANTS[model] in ("NoPE", "SDP") else "o")
    for model in MODEL_ORDER
}


def model_legend_handles() -> list[plt.Line2D]:
    return [
        plt.Line2D([0], [0], color=MODEL_COLORS[m], marker=MODEL_MARKERS[m], label=m)
        for m in MODEL_ORDER
    ]


def add_side_legend(fig: plt.Figure) -> None:
    """Model legend on the right of the panels, matching the dt figures."""
    fig.legend(
        handles=model_legend_handles(),
        loc="center left",
        bbox_to_anchor=(0.92, 0.5),
        frameon=False,
    )
# Hue encodes architecture and parameter count sets the shade, via the shared
# raised-floor ramp in aesthetics: 275M models take the same tone as the
# directly-trained 275M models in dt_results, and larger sizes (including
# off-anchor ones like 450M or 2.7B) interpolate monotonically darker.
MODEL_COLORS = {
    model: aes.model_size_color_millions(MODEL_ARCHITECTURES[model], MODEL_SIZES[model])
    for model in MODEL_ORDER
}
COMPARISON_MODELS = [
    model
    for model in MODEL_ORDER
    if MODEL_ARCHITECTURES[model] in COMPARISON_ARCHITECTURE_ORDER
]
item_df["complexity"] = pd.Categorical(
    item_df["complexity"],
    [value.replace("_", "-") for value in COMPLEXITY_ORDER],
    ordered=True,
)
missing_metadata = item_df[JOINED_METADATA_COLUMNS].isna().any(axis=1).sum()
if missing_metadata:
    print(
        f"Warning: {missing_metadata:,} rows lack joined dataset metadata "
        f"(dependency length / sensitivity fields)"
    )
print(
    f"Loaded {len(item_df):,} example/model rows across {item_df['task'].nunique()} task variants"
)
display(
    item_df.groupby("model", observed=True).agg(
        examples=("score", "size"),
        accuracy=("score", "mean"),
        min_tokens=("prompt_tokens", "min"),
        max_tokens=("prompt_tokens", "max"),
    )
)

In [ ]:
item_df["domain"].unique()

In [ ]:
item_df["num_variables"].unique()

## Accuracy vs prompt length

Points use power-of-two prompt-token bins that automatically span the observed data; bands are Wilson 95% intervals. Bins with fewer than `MIN_BIN_EXAMPLES` examples for an architecture are omitted.

In [ ]:
def wilson_interval(successes: pd.Series, n: pd.Series) -> tuple[pd.Series, pd.Series]:
    z = 1.96
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    radius = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return center - radius, center + radius


length_min_power = math.floor(math.log2(item_df["prompt_tokens"].min()))
length_max_power = math.floor(math.log2(item_df["prompt_tokens"].max())) + 1
LENGTH_EDGES = 2.0 ** np.arange(length_min_power, length_max_power + 1)

item_df["length_bin"] = pd.cut(
    item_df["prompt_tokens"], LENGTH_EDGES, right=False, include_lowest=True
)
item_df["length_mid"] = (
    item_df["length_bin"]
    .map(
        lambda value: math.sqrt(value.left * value.right) if pd.notna(value) else np.nan
    )
    .astype(float)
)
length_df = item_df.groupby(
    ["domain", "complexity", "model", "length_bin", "length_mid"],
    observed=True,
    as_index=False,
).agg(
    accuracy=("score", "mean"),
    successes=("score", "sum"),
    n=("score", "size"),
    chance_total=("chance", "sum"),
    chance_examples=("chance", "size"),
)
length_df = length_df[length_df["n"] >= MIN_BIN_EXAMPLES].copy()
length_df["ci_low"], length_df["ci_high"] = wilson_interval(
    length_df["successes"], length_df["n"]
)


def configure_length_axis(ax: plt.Axes) -> None:
    ax.set_xscale("log", base=10)
    ax.set_xlim(LENGTH_EDGES[0], LENGTH_EDGES[-1])
    for context_length, color in CONTEXT_REFERENCE_LINES.items():
        ax.axvline(
            context_length,
            color=color,
            linestyle=":",
            linewidth=0.8,
            zorder=1,
        )
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10, numticks=8))
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.xaxis.set_minor_locator(
        mtick.LogLocator(base=10, subs=(1,2,3,4,5,6,7,8,9), numticks=100)
    )
    ax.xaxis.set_minor_formatter(mtick.NullFormatter())


FORMAT_ORDER = [
    "Cube Painting",
    "Integer Code",
    "People In Rooms",
    "Spreadsheet Cells",
    "Status Lights",
    "Ruler",
]
MONOID_COMPLEXITY_ORDER = ["r-trivial", "aperiodic", "periodic"]
fig, axes = plt.subplots(
    len(FORMAT_ORDER),
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, 8),
    sharex=True,
    sharey=True,
)
for row_idx, formatting in enumerate(FORMAT_ORDER):
    for col_idx, complexity in enumerate(MONOID_COMPLEXITY_ORDER):
        ax = axes[row_idx, col_idx]
        subset = length_df[
            (length_df["domain"] == formatting)
            & (length_df["complexity"] == complexity)
        ]
        for model in MODEL_ORDER:
            curve = subset[subset["model"] == model].sort_values("length_mid")
            x = curve["length_mid"].to_numpy(float)
            y = curve["accuracy"].to_numpy(float)
            ax.plot(
                x, y, marker=MODEL_MARKERS[model], ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
            )
            ax.fill_between(
                x,
                curve["ci_low"].to_numpy(float),
                curve["ci_high"].to_numpy(float),
                color=MODEL_COLORS[model],
                alpha=0.12,
            )
        add_chance_line(ax, subset)
        configure_length_axis(ax)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        if row_idx == 0:
            ax.set_title(complexity.title(), fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(formatting, fontsize=9)
axes[-1,0].set_xlabel("Context length (log scale)", fontsize=10)
# fig.supxlabel("Context length (log scale)")
add_side_legend(fig)
# fig.suptitle("Accuracy vs prompt length by task", y=1.01)
fig.tight_layout()
_ = aes.save_figure(FIGURES_DIR / "pt_accuracy_by_context_length", fig=fig, save_png=True)
plt.show()

In [ ]:
summary_length_df = (
    item_df.groupby(
        ["complexity", "model", "length_bin", "length_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
summary_length_df["ci_low"], summary_length_df["ci_high"] = wilson_interval(
    summary_length_df["successes"], summary_length_df["n"]
)

fig, axes = plt.subplots(
    1,
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex=True,
    sharey=True,
)
for ax, complexity in zip(axes, MONOID_COMPLEXITY_ORDER, strict=True):
    subset = summary_length_df[summary_length_df["complexity"] == complexity]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        y = curve["accuracy"].to_numpy(float)
        ax.plot(
            x, y, marker=MODEL_MARKERS[model], ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    add_chance_line(ax, subset)
    configure_length_axis(ax)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(complexity.title(), fontsize=9)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Context length (log scale)")
# fig.supxlabel("Context length (log scale)")
add_side_legend(fig)
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_context_length_aggregated",
    fig=fig,
    save_png=True,
)
plt.show()

## Accuracy vs number of assignments

Same pooling and binning approach as the prompt-length plots above, but bucketed by `num_assignments` (the number of assignment operations in the trace) instead of context length. Bins are powers of two spanning the observed range; first per formatter, then pooled across formatting.


In [ ]:
assignment_min_power = math.floor(math.log2(item_df["num_assignments"].min()))
assignment_max_power = math.floor(math.log2(item_df["num_assignments"].max())) + 1
ASSIGNMENT_EDGES = 2.0 ** np.arange(assignment_min_power, assignment_max_power + 1)

item_df["assignment_bin"] = pd.cut(
    item_df["num_assignments"], ASSIGNMENT_EDGES, right=False, include_lowest=True
)
item_df["assignment_mid"] = (
    item_df["assignment_bin"]
    .map(
        lambda value: math.sqrt(value.left * value.right) if pd.notna(value) else np.nan
    )
    .astype(float)
)


def configure_assignment_axis(ax: plt.Axes) -> None:
    ax.set_xscale("log", base=10)
    ax.set_xlim(ASSIGNMENT_EDGES[0], ASSIGNMENT_EDGES[-1])
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10, numticks=8))
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.xaxis.set_minor_locator(
        mtick.LogLocator(base=10, subs=(1,2,3,4,5,6,7,8,9), numticks=100)
    )
    ax.xaxis.set_minor_formatter(mtick.NullFormatter())


assignment_df = item_df.groupby(
    ["domain", "complexity", "model", "assignment_bin", "assignment_mid"],
    observed=True,
    as_index=False,
).agg(
    accuracy=("score", "mean"),
    successes=("score", "sum"),
    n=("score", "size"),
    chance_total=("chance", "sum"),
    chance_examples=("chance", "size"),
)
assignment_df = assignment_df[assignment_df["n"] >= MIN_BIN_EXAMPLES].copy()
assignment_df["ci_low"], assignment_df["ci_high"] = wilson_interval(
    assignment_df["successes"], assignment_df["n"]
)

fig, axes = plt.subplots(
    len(FORMAT_ORDER),
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, 8),
    sharex=True,
    sharey=True,
)
for row_idx, formatting in enumerate(FORMAT_ORDER):
    for col_idx, complexity in enumerate(MONOID_COMPLEXITY_ORDER):
        ax = axes[row_idx, col_idx]
        subset = assignment_df[
            (assignment_df["domain"] == formatting)
            & (assignment_df["complexity"] == complexity)
        ]
        for model in MODEL_ORDER:
            curve = subset[subset["model"] == model].sort_values("assignment_mid")
            x = curve["assignment_mid"].to_numpy(float)
            y = curve["accuracy"].to_numpy(float)
            ax.plot(
                x, y, marker=MODEL_MARKERS[model], ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
            )
            ax.fill_between(
                x,
                curve["ci_low"].to_numpy(float),
                curve["ci_high"].to_numpy(float),
                color=MODEL_COLORS[model],
                alpha=0.12,
            )
        add_chance_line(ax, subset)
        configure_assignment_axis(ax)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        if row_idx == 0:
            ax.set_title(complexity.title(), fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(formatting, fontsize=9)
axes[-1, 0].set_xlabel("# assignments (log scale)", fontsize=10)
add_side_legend(fig)
fig.tight_layout()
_ = aes.save_figure(FIGURES_DIR / "pt_accuracy_by_assignments", fig=fig, save_png=True)
plt.show()


In [ ]:
summary_assignment_df = (
    item_df.groupby(
        ["complexity", "model", "assignment_bin", "assignment_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
summary_assignment_df["ci_low"], summary_assignment_df["ci_high"] = wilson_interval(
    summary_assignment_df["successes"], summary_assignment_df["n"]
)

fig, axes = plt.subplots(
    1,
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex=True,
    sharey=True,
)
for ax, complexity in zip(axes, MONOID_COMPLEXITY_ORDER, strict=True):
    subset = summary_assignment_df[summary_assignment_df["complexity"] == complexity]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("assignment_mid")
        x = curve["assignment_mid"].to_numpy(float)
        y = curve["accuracy"].to_numpy(float)
        ax.plot(
            x, y, marker=MODEL_MARKERS[model], ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    add_chance_line(ax, subset)
    configure_assignment_axis(ax)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(complexity.title(), fontsize=9)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("# assignments (log scale)")
add_side_legend(fig)
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_assignments_aggregated",
    fig=fig,
    save_png=True,
)
plt.show()

## Periodic accuracy vs dependency length

Dependency length uses the periodic sample's dataset metadata selected by `DEPENDENCY_LENGTH_FIELD` (by default, `max_dependency_length`). The saved evaluation request documents omit these fields, so the load step joins them from the source dataset using each example's native ID. This view pools examples across all five formatting variants into approximately powers-of-two dependency-length bins; points show pooled accuracy and bands show Wilson 95% intervals.

In [ ]:
periodic_items = item_df[item_df["complexity"] == "periodic"].copy()
periodic_items["dependency_bin"] = pd.cut(
    periodic_items["dependency_length"],
    DEPENDENCY_EDGES,
    right=False,
    include_lowest=True,
)
periodic_dependency_df = (
    periodic_items.groupby(
        ["dependency_bin", "model"], observed=True, as_index=False
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
        dependency_mid=("dependency_length", "median"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
)
periodic_dependency_df["ci_low"], periodic_dependency_df["ci_high"] = (
    wilson_interval(
        periodic_dependency_df["successes"], periodic_dependency_df["n"]
    )
)

fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
for model in MODEL_ORDER:
    curve = periodic_dependency_df[
        periodic_dependency_df["model"] == model
    ].sort_values("dependency_mid")
    x = curve["dependency_mid"].to_numpy(float)
    ax.plot(
        x,
        curve["accuracy"].to_numpy(float),
        marker=MODEL_MARKERS[model],
        ms=3,
        lw=1.5,
        color=MODEL_COLORS[model],
        label=model,
    )
    ax.fill_between(
        x,
        curve["ci_low"].to_numpy(float),
        curve["ci_high"].to_numpy(float),
        color=MODEL_COLORS[model],
        alpha=0.12,
    )
add_chance_line(ax, periodic_dependency_df)
ax.set_xscale("symlog", base=10, linthresh=1)
ax.set_xlim(0, None)
minor_dependency_ticks = (
    np.arange(2, 10)[None, :] * 10 ** np.arange(4)[:, None]
).ravel()
ax.set_xticks(minor_dependency_ticks, minor=True)
ax.tick_params(axis="x", which="minor", length=3)
ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
ax.set(
    xlabel="Maximum dependency length",
    ylabel="Accuracy",
    ylim=(0, 1.02),
)
ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.grid(alpha=0.25)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
sns.despine()
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_dependency_length",
    fig=fig,
    save_png=True,
)
plt.show()

## Periodic accuracy vs sensitivity density

`sensitivity_density` is per-example metadata from the source dataset (`state_bench.semantics.sensitivity_summary`). At each semantic step of the trace it does a reverse dataflow pass to find, for every state variable, how many of the queried outputs would change if that variable's value changed at that point; `sensitivity_density` is that count averaged over all (variable, output, step) triples. Higher density means more of the tracked state stays causally load-bearing for the final answer throughout the trace, rather than being safely ignorable filler. `sensitivity_coverage` (fraction of variables live at each step) and `mean_live_fanout` (outputs affected per live variable) decompose the same quantity and are also joined below. This metadata is populated for every complexity, but r-trivial and aperiodic traces are sensitivity-poor by construction (density is near zero), so this view is restricted to periodic examples, pooled across formatting and binned by `SENSITIVITY_DENSITY_EDGES`.

In [ ]:
sensitivity_items = item_df[
    (item_df["complexity"] == "periodic") & item_df["sensitivity_density"].notna()
].copy()
sensitivity_items["sensitivity_bin"] = pd.cut(
    sensitivity_items["sensitivity_density"],
    SENSITIVITY_DENSITY_EDGES,
    include_lowest=True,
)
sensitivity_df = (
    sensitivity_items.groupby(
        ["sensitivity_bin", "model"], observed=True, as_index=False
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
        sensitivity_mid=("sensitivity_density", "median"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
)
sensitivity_df["ci_low"], sensitivity_df["ci_high"] = wilson_interval(
    sensitivity_df["successes"], sensitivity_df["n"]
)

fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
for model in MODEL_ORDER:
    curve = sensitivity_df[sensitivity_df["model"] == model].sort_values(
        "sensitivity_mid"
    )
    x = curve["sensitivity_mid"].to_numpy(float)
    ax.plot(
        x,
        curve["accuracy"].to_numpy(float),
        marker=MODEL_MARKERS[model],
        ms=3,
        lw=1.5,
        color=MODEL_COLORS[model],
        label=model,
    )
    ax.fill_between(
        x,
        curve["ci_low"].to_numpy(float),
        curve["ci_high"].to_numpy(float),
        color=MODEL_COLORS[model],
        alpha=0.12,
    )
add_chance_line(ax, sensitivity_df)
ax.set(
    xlabel="Sensitivity density",
    ylabel="Accuracy",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.grid(alpha=0.25)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
sns.despine()
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_sensitivity",
    fig=fig,
    save_png=True,
)
plt.show()

## Periodic accuracy by number of variables

Periodic examples are pooled across formatting and split into panels for 2, 4, and 6 variables. Context-length bins and reference lines are shared with the other prompt-length plots.

In [ ]:
VARIABLE_COUNTS = [2, 4, 8, 16]
periodic_variable_df = (
    item_df[
        (item_df["complexity"] == "periodic")
        & item_df["num_variables"].isin(VARIABLE_COUNTS)
    ]
    .groupby(
        ["num_variables", "model", "length_bin", "length_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
periodic_variable_df["ci_low"], periodic_variable_df["ci_high"] = (
    wilson_interval(
        periodic_variable_df["successes"], periodic_variable_df["n"]
    )
)

fig, axes = plt.subplots(
    1,
    len(VARIABLE_COUNTS),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex=True,
    sharey=True,
)
for ax, num_variables in zip(axes, VARIABLE_COUNTS, strict=True):
    subset = periodic_variable_df[
        periodic_variable_df["num_variables"] == num_variables
    ]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        ax.plot(
            x,
            curve["accuracy"].to_numpy(float),
            marker=MODEL_MARKERS[model],
            ms=2.5,
            lw=1.2,
            color=MODEL_COLORS[model],
            label=model,
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    add_chance_line(ax, subset)
    configure_length_axis(ax)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(f"{num_variables} variables", fontsize=9)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Context length (log scale)")
add_side_legend(fig)
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_num_variables_periodic",
    fig=fig,
    save_png=True,
)
plt.show()

In [ ]:
VARIABLE_COUNTS = [2, 4, 8, 16]
r_trivial_variable_df = (
    item_df[
        (item_df["complexity"] == "r-trivial")
        & item_df["num_variables"].isin(VARIABLE_COUNTS)
    ]
    .groupby(
        ["num_variables", "model", "length_bin", "length_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
r_trivial_variable_df["ci_low"], r_trivial_variable_df["ci_high"] = (
    wilson_interval(
        r_trivial_variable_df["successes"], r_trivial_variable_df["n"]
    )
)

fig, axes = plt.subplots(
    1,
    len(VARIABLE_COUNTS),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex=True,
    sharey=True,
)
for ax, num_variables in zip(axes, VARIABLE_COUNTS, strict=True):
    subset = r_trivial_variable_df[
        r_trivial_variable_df["num_variables"] == num_variables
    ]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        ax.plot(
            x,
            curve["accuracy"].to_numpy(float),
            marker=MODEL_MARKERS[model],
            ms=2.5,
            lw=1.2,
            color=MODEL_COLORS[model],
            label=model,
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    add_chance_line(ax, subset)
    configure_length_axis(ax)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(f"{num_variables} variables", fontsize=9)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Context length (log scale)")
add_side_legend(fig)
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_num_variables_r_trivial",
    fig=fig,
    save_png=True,
)
plt.show()

In [ ]:
VARIABLE_COUNTS = [2, 4, 8, 16]
aperiodic_variable_df = (
    item_df[
        (item_df["complexity"] == "aperiodic")
        & item_df["num_variables"].isin(VARIABLE_COUNTS)
    ]
    .groupby(
        ["num_variables", "model", "length_bin", "length_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
        chance_total=("chance", "sum"),
        chance_examples=("chance", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
aperiodic_variable_df["ci_low"], aperiodic_variable_df["ci_high"] = (
    wilson_interval(
        aperiodic_variable_df["successes"], aperiodic_variable_df["n"]
    )
)

fig, axes = plt.subplots(
    1,
    len(VARIABLE_COUNTS),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex=True,
    sharey=True,
)
for ax, num_variables in zip(axes, VARIABLE_COUNTS, strict=True):
    subset = aperiodic_variable_df[
        aperiodic_variable_df["num_variables"] == num_variables
    ]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        ax.plot(
            x,
            curve["accuracy"].to_numpy(float),
            marker=MODEL_MARKERS[model],
            ms=2.5,
            lw=1.2,
            color=MODEL_COLORS[model],
            label=model,
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    add_chance_line(ax, subset)
    configure_length_axis(ax)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(f"{num_variables} variables", fontsize=9)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Context length (log scale)")
add_side_legend(fig)
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_num_variables_aperiodic",
    fig=fig,
    save_png=True,
)
plt.show()

In [ ]:
periodic_variable_df

## R-trivial breakout B: padding complexity

Padding complexity is the dataset's `filler_distraction` level (`easy`, `medium`, or `hard`). The table marginalizes over padding amount. The context-length panels retain each padding amount separately and repeat the unpadded control as the first point in each panel.

In [ ]:
PADDING_COMPLEXITY_ORDER = ["none", "easy", "medium", "hard"]
TASK_COMPLEXITY_ORDER = ["r-trivial", "aperiodic", "periodic"]
PLOTTED_PADDING_COMPLEXITIES = ["easy", "medium", "hard"]
padding_items = item_df[item_df["padding_complexity"].notna()].copy()
padding_items["padding_complexity"] = pd.Categorical(
    padding_items["padding_complexity"], PADDING_COMPLEXITY_ORDER, ordered=True
)
padding_curve_df = padding_items[
    padding_items["padding_complexity"] != "none"
].groupby(
    [
        "complexity",
        "padding_complexity",
        "length_bin",
        "length_mid",
        "model",
    ],
    observed=True,
    as_index=False,
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    chance_total=("chance", "sum"),
    chance_examples=("chance", "size"),
)
padding_curve_df = padding_curve_df[
    padding_curve_df["n"] >= MIN_BIN_EXAMPLES
].copy()
fig, axes = plt.subplots(
    len(PLOTTED_PADDING_COMPLEXITIES),
    len(TASK_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, 5),
    sharex=True,
    sharey=True,
)
for row_idx, padding_complexity in enumerate(PLOTTED_PADDING_COMPLEXITIES):
    for col_idx, task_complexity in enumerate(TASK_COMPLEXITY_ORDER):
        ax = axes[row_idx, col_idx]
        subset = padding_curve_df[
            (padding_curve_df["padding_complexity"] == padding_complexity)
            & (padding_curve_df["complexity"] == task_complexity)
        ]
        for model in MODEL_ORDER:
            curve = subset[subset["model"] == model].sort_values("length_mid")
            ax.plot(
                curve["length_mid"],
                curve["accuracy"],
                marker=MODEL_MARKERS[model],
                lw=1.2,
                ms=2.5,
                color=MODEL_COLORS[model],
                label=model,
            )
        add_chance_line(ax, subset)
        configure_length_axis(ax)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.grid(alpha=0.25)
        if row_idx == 0:
            ax.set_title(task_complexity.title())
        if col_idx == 0:
            ax.set_ylabel(f"{padding_complexity.title()}\nAccuracy")

axes[-1, 0].set_xlabel("Context length (log scale)")
add_side_legend(fig)
fig.suptitle("Accuracy vs context length by task and padding complexity", y=1.02)
sns.despine()
fig.tight_layout()
_ = aes.save_figure(
    FIGURES_DIR / "pt_accuracy_by_padding_complexity", fig=fig, save_png=True
)
plt.show()